#### 

In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils import data
from torch_geometric.data import Data
from scipy.sparse import issparse
from torchvision import transforms
import cv2
import scanpy as sc

from utils import load_ST_file

# 传入的 csv_file 必须是包含那 314 个通路的 KEGG 数据库文件。
# 传入的 all_genes_list 必须是你 adata.var_names 的顺序。
class PathwayProcessor:
    """
    Optimized pathway processor using index-based approach.
    Pre-computes pathway gene indices and edge structures for fast runtime access.
    """
    def __init__(self, csv_file, all_genes_list):
        df = pd.read_csv(csv_file)
        df = df[df['EdgeInfo'].notnull()]

        # Create gene name to index mapping
        self.all_genes_map = {gene: i for i, gene in enumerate(all_genes_list)}

        self.pathway_ids = []
        self.pathway_gene_indices = []  # Each pathway as list of gene indices
        self.pathway_edge_indices = []  # Pre-computed edge_index tensors

        print("Pre-computing pathway structures...")
        for _, row in df.iterrows():
            pathway_id = row['PathwayID']
            self.pathway_ids.append(pathway_id)

            # Parse nodes and convert to indices
            nodes = [node.strip() for node in row['NodeInfo'].split(';')]
            gene_indices = [self.all_genes_map[gene] for gene in nodes if gene in self.all_genes_map]
            self.pathway_gene_indices.append(gene_indices)

            # Parse edges and build edge_index
            edge_info = row['EdgeInfo']
            if pd.notna(edge_info) and len(gene_indices) > 0:
                # Create node to local index mapping for this pathway
                # IMPORTANT: Only assign indices to genes that exist in all_genes_map
                valid_nodes = [gene for gene in nodes if gene in self.all_genes_map]
                node_to_local_idx = {gene: idx for idx, gene in enumerate(valid_nodes)}

                edges = []
                for edge in edge_info.split(';'):
                    if edge.strip():
                        try:
                            # Parse edge format: "(gene1, gene2)"
                            edge_clean = edge.strip('() ')
                            src, dst = edge_clean.split(',')
                            src, dst = src.strip(), dst.strip()

                            if src in node_to_local_idx and dst in node_to_local_idx:
                                edges.append([node_to_local_idx[src], node_to_local_idx[dst]])
                        except:
                            continue

                if edges:
                    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
                else:
                    # No valid edges, create empty edge_index
                    edge_index = torch.empty((2, 0), dtype=torch.long)
            else:
                # No edges or no valid genes
                edge_index = torch.empty((2, 0), dtype=torch.long)

            self.pathway_edge_indices.append(edge_index)

        print(f"Pre-computed {len(self.pathway_ids)} pathways")

    def get_num_pathways(self):
        return len(self.pathway_ids)

In [2]:
import torch
from torch_geometric.data import Data, Batch
import numpy as np
from tqdm import tqdm

def get_pathway_embeddings(adata, processor, model, device='cuda'):
    """
    使用预训练模型提取每个 Spot 的通路嵌入向量
    """
    model.to(device)
    model.eval()
    
    # 确保 adata 的基因顺序与 processor 初始化时的 all_genes_list 一致
    # 假设 adata 已经过 normalize_total 和 log1p 处理
    
    all_spot_embeddings = []
    
    print("开始提取特征...")
    with torch.no_grad():
        for i in tqdm(range(adata.n_obs)):
            # 1. 提取当前 Spot 的所有通路图
            spot_gene_exp = adata.X[i].toarray().flatten() if issparse(adata.X) else adata.X[i].flatten()
            
            xg_list = []
            for p_idx in range(processor.get_num_pathways()):
                # 获取该通路的基因索引和边
                gene_indices = processor.pathway_gene_indices[p_idx]
                edge_index = processor.pathway_edge_indices[p_idx]
                
                if len(gene_indices) > 0:
                    # 提取该通路下基因的表达值作为节点特征 [num_nodes, 1]
                    x = torch.tensor(spot_gene_exp[gene_indices], dtype=torch.float).unsqueeze(1)
                    # 构建 PyG 的 Data 对象
                    g = Data(x=x, edge_index=edge_index)
                else:
                    # 如果通路为空，建立一个虚拟节点（保持维度一致）
                    g = Data(x=torch.zeros((1, 1)), edge_index=torch.empty((2, 0), dtype=torch.long))
                xg_list.append(g)
            
            # 2. 将 314 个图合并为一个 Batch 提高计算效率
            # 注意：SpaCLR 内部可能期望 List[Batch]，这里根据你模型 forward 的需求调整
            # 这里的处理逻辑需要对应 model.forward_gene
            xg_batch_list = [Batch.from_data_list([g]).to(device) for g in xg_list]
            
            # 3. 推理
            # aggregated_emb 是融合后的向量，hg 是 314 个通路各自的嵌入
            aggregated_emb, hg, attn_weights = model.forward_gene(xg_batch_list)
            
            # 4. 收集结果
            # 如果你需要的是融合后的 16/32 维特征：用 aggregated_emb
            # 如果你需要的是 314 维通路原始特征：从 hg 中提取
            all_spot_embeddings.append(aggregated_emb.cpu().numpy())
            
    return np.vstack(all_spot_embeddings)

#### model

In [6]:
class SpaCLR(nn.Module):
    def __init__(self, num_node_features, hidden_dim, last_dim, image_dims, p_drop, n_pos, backbone='densenet', projection_dims=[64, 64]):
        super(SpaCLR, self).__init__()
        self.gene_encoder = GraphSAGE(num_node_features, hidden_dim, last_dim)
        self.set2set = Set2Set(last_dim, processing_steps=3)
        self.set2set_linear = nn.Linear(2 * last_dim, last_dim)
        self.attention_layer = AttentionLayer(last_dim)
        self.mse_loss = nn.MSELoss()

        if backbone == 'densenet':
            self.image_encoder = densenet121(pretrained=True)
            n_features = self.image_encoder.classifier.in_features
            self.image_encoder.classifier = nn.Identity()
        elif backbone == 'resnet':
            self.image_encoder = resnet50(pretrained=True)
            n_features = self.image_encoder.fc.in_features
            self.image_encoder.fc = nn.Identity()

        self.x_embedding = nn.Embedding(n_pos, n_features)
        self.y_embedding = nn.Embedding(n_pos, n_features)

        image_dims[0] = n_features
        image_dims.append(projection_dims[0])
        self.image_linear = nn.Sequential(OrderedDict([
            (f'image_block{i+1}', LinearBlock(image_dims[i], image_dims[i+1], p_drop)) for i, _ in enumerate(image_dims[:-1])
        ]))

        self.projector = nn.Sequential(
            nn.Linear(projection_dims[0], projection_dims[0]),
            nn.ReLU(),
            nn.Linear(projection_dims[0], projection_dims[1]),
        )

    def forward_image(self, xi, spatial):
        xi = self.image_encoder(xi)
        xi = self.image_linear(xi)
        hi = self.projector(xi)
        return xi, hi

    def forward_gene(self, xg_list):
        all_embeddings = []
        all_batches = []

        # Process all graphs in one go
        for xg_batch in xg_list:
            embeddings = self.gene_encoder(xg_batch)
            all_embeddings.append(embeddings)
            all_batches.append(xg_batch.batch)

        # Apply set2set to all embeddings
        graph_embeddings_list = []
        for embeddings, batch in zip(all_embeddings, all_batches):
            graph_embeddings = self.set2set(embeddings, batch)
            graph_embeddings = self.set2set_linear(graph_embeddings)
            graph_embeddings_list.append(graph_embeddings)

        all_embeddings = torch.stack(graph_embeddings_list)
        all_embeddings = all_embeddings.transpose(0, 1)  # [B, N_pathways, F] -> [N_pathways, B, F]
        aggregated_embeddings, attention_weights = self.attention_layer(all_embeddings)
        hg = self.projector(aggregated_embeddings)
        return aggregated_embeddings, hg, attention_weights

    def forward(self, xg_list, xi, spatial):
        xg, hg, pathway_attn_weights = self.forward_gene(xg_list)
        xi, hi = self.forward_image(xi, spatial)
        return xg, xi, hg, hi, pathway_attn_weights


In [7]:
# --- 配置环境 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
csv_path = "/data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY/GeneSets/HSA_KEGG_Pathway.csv"  # 你的通路数据库文件
weight_path = "/data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY/GeneSets/PathCLAST_IDC_weight.pth" # 权重文件
# /data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY/kz_data/Raw_Data/h5ad_files/
# --- 1. 加载并清洗数据 ---
adata = load_ST_file("/data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY/kz_data/Raw_Data/h5ad_files/151673.h5ad") # 使用你提供的 utils 函数
# 必须保证基因列表与训练时一致
all_genes = adata.var_names.tolist()

# --- 2. 初始化预处理器 ---
processor = PathwayProcessor(csv_path, all_genes)

# --- 3. 初始化模型并加载权重 ---
# 这里的参数 (hidden_dim等) 必须参考原论文训练配置
model = SpaCLR(num_node_features=1, hidden_dim=32, last_dim=16, p_drop=0.1, n_pos) 

# 加载权重
state_dict = torch.load(weight_path, map_location=device)
model.load_state_dict(state_dict)

# --- 4. 提取嵌入向量 ---
embeddings = get_pathway_embeddings(adata, processor, model, device)

# --- 5. 后续处理 (聚类) ---
# 将结果存入 adata
adata.obsm['X_pathclast'] = embeddings
sc.pp.neighbors(adata, use_rep='X_pathclast')
sc.tl.leiden(adata) # 或者使用你代码里的 mclust_R

Pre-computing pathway structures...
Pre-computed 314 pathways


TypeError: __init__() missing 3 required positional arguments: 'image_dims', 'p_drop', and 'n_pos'